In [217]:
import os 
from dotenv import load_dotenv
import json
from langchain_gigachat.chat_models import GigaChat
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers import JsonOutputParser

load_dotenv()

# Настройка модели GigaChat с расширенными параметрами
llm = GigaChat(
    credentials=os.getenv('GIGA_KEY'),
    model='GigaChat-2', 
    verify_ssl_certs=False,
    temperature=0.2, # настройка креативности
    max_tokens=1000 # максимальная длина ответа
)

# Проверка подключения
# response = llm.invoke('Привет! Как дела?')
# print(response.content)

In [218]:
# Определяем JSON-схему вручную
json_schema = {
    "type": "object",
    "properties": {
        "total_persons": {"type": "integer", "description": "Общее количество человек"},
        "count_adults": {"type": "integer", "description": "Количество взрослых"},
        "count_children": {"type": "integer", "description": "Количество детей"},
        "start_date": {"type": "string", "description": "Дата заезда"},
        "nights": {"type": "integer", "description": "Количество ночей проживания"},
        "price_per_day": {"type": "integer", "description": "Желаемая цена за сутки"},
        "remarks": {"type": "string", "description": "Особые пожелания"}
    },
    "required": ["total_persons", "count_adults", "count_children", "start_date", "nights", "price_per_day"]
}

# Создаем парсер для JSON
json_parser = JsonOutputParser(schema=json_schema)

In [219]:
# Собираем промпт с детализированными инструкциями для модели
system_prompt = SystemMessagePromptTemplate.from_template("""
Ты — эксперт по анализу текстов заявок на аренду жилья.

Твоя задача — извлечь из заявки данные:
- total_persons — общее количество проживающих;
- count_adults — количество взрослых;
- count_children — количество детей;
- start_date — дата заезда;
- nights — количество ночей;
- price_per_day — желаемая цена за сутки;
- remarks — особые пожелания.

ДЕТАЛИЗИРОВАННЫЕ ИНСТРУКЦИИ:
1. Внимательно прочитай текст заявки.
2. Найди все упоминания количества людей: взрослых, детей, семьи, пары, одного человека.
3. Используй правила:
   - "семья с одним ребенком" = 3 человека: 2 взрослых и 1 ребенок;
   - "семья с двумя детьми" = 4 человека: 2 взрослых и 2 ребенка;
   - "пара", "двое", "молодая семья" = 2 взрослых;
   - "один", "студент", "индивидуально" = 1 взрослый;                                                 
4. Правила для определения даты заезда:
   - Если даты представлены в диапазоне, бери самую раннюю дату заезда;
   - Возвращай дату в формате "ДД.ММ", год указывать не нужно;
   - Если указано конец месяца, возвращай последний день этого месяца;
   - Если указано начало месяца, возвращай первый день этого месяца;
   - Если указан месяц без числа, возвращай 1 число этого месяца.                                                      
   Примеры:
   - "03.03" = "03.03";
   - "с 15 июля" = "15.07";
   - "26.02.2020" = "26.02";
   - "конец марта" = "31.03";
   - "начало мая" = "01.05";  
   - "январь" = "01.01".                                              
5. ОБРАТИ ВНИМАНИЕ на правила для определения количества ночей:
   - Если указано ДИАПАЗОН дат, рассчитывай количество ночей как разницу между датами (например, "c 10 мая по 15 мая" = 5 ночей, "с 23.07 по 04.08" = 12 ночей);
   - Если указано количество ДНЕЙ, возвращай на 1 ночь МЕНЬШЕ (например, "на 7 дней" = 6 ночей);
   - Если указано количество ночей, возвращай это число без изменений (например, "на 5 ночей" = 5 ночей);
   - Если указано "на неделю", возвращай 7 ночей (например, "на неделю" = 7 ночей, "на 2 недели" = 14 ночей);
6. Если цена указана диапазоном, бери максимальную.
7. Если поле не указано:
   - для чисел обязательно верни 0 - это поля: total_persons, count_adults, count_children, nights, price_per_day.;
   - для текста обязательно верни пустую строку "" - это поля: start_date, remarks.
8. Первый символ ответа должен быть открывающейся фигурной скобкой.
9. Последний символ ответа должен быть закрывающейся фигурной скобкой.
""")

user_prompt = HumanMessagePromptTemplate.from_template("""
Проанализируй заявку на аренду жилья:

{text}

Верни только JSON-объект строго по инструкции ниже.

{format_instructions}
""")

chat_prompt = ChatPromptTemplate.from_messages([system_prompt, user_prompt])

In [220]:
import pandas as pd

# Загрузка данных из csv в датафрейм df
df = pd.read_csv('rental_09.csv', sep=';', dtype={"start_date": str})

In [221]:
# Создание цепочки и вызов
responses_list = []

chain = chat_prompt | llm | json_parser
for _, row in df.iterrows():
    text = row['text']
    response: dict = chain.invoke({'text': text, 'format_instructions': json_parser.get_format_instructions()})
    responses_list.append(response)
    print(json.dumps(response, ensure_ascii=False, indent=2))

{
  "total_persons": 6,
  "count_adults": 3,
  "count_children": 3,
  "start_date": "30.07",
  "nights": 10,
  "price_per_day": 0,
  "remarks": ""
}
{
  "total_persons": 3,
  "count_adults": 2,
  "count_children": 1,
  "start_date": "12.07",
  "nights": 5,
  "price_per_day": 0,
  "remarks": ""
}
{
  "total_persons": 2,
  "count_adults": 2,
  "start_date": "03.09",
  "nights": 6,
  "price_per_day": 0,
  "count_children": 0,
  "remarks": ""
}
{
  "total_persons": 4,
  "count_adults": 4,
  "start_date": "31.07",
  "nights": 7,
  "price_per_day": 0,
  "count_children": 0,
  "remarks": ""
}
{
  "total_persons": 3,
  "count_adults": 3,
  "count_children": 0,
  "start_date": "13.06",
  "nights": 8,
  "price_per_day": 0,
  "remarks": ""
}
{
  "total_persons": 2,
  "count_adults": 2,
  "start_date": "07.09",
  "nights": 7,
  "price_per_day": 700,
  "count_children": 0,
  "remarks": ""
}
{
  "total_persons": 3,
  "count_adults": 2,
  "count_children": 1,
  "start_date": "08.07",
  "nights": 14,


In [ ]:
responses_df = pd.DataFrame(responses_list)

fields_to_check = ["total_persons", "count_adults", "count_children", "start_date", "nights", "price_per_day"]

accuracy_by_field = {}

for field in fields_to_check:
    correct = 0
    total = len(df)
    for i in range(total):
        true_value = df.loc[i, field]
        resp_value = responses_df.loc[i, field]
        if field in ["total_persons", "count_adults", "count_children", "nights", "price_per_day"]:
            true_value = pd.to_numeric(true_value, errors="coerce")
            resp_value = pd.to_numeric(resp_value, errors="coerce")
            if true_value == resp_value:
                correct += 1
        elif field == "start_date":
            true_value = str(true_value)
            resp_value = str(resp_value)
            if true_value == resp_value:
                correct += 1
    accuracy = correct / total
    accuracy_by_field[field] = accuracy

    print(f"{field}: ошибок: {total - correct}, точность: {accuracy:.1%}")

average_accuracy = sum(accuracy_by_field.values()) / len(accuracy_by_field)

print(f"Средняя точность по всем показателям: {average_accuracy:.1%}")

total_persons: ошибок: 0, точность: 100.0%
count_adults: ошибок: 0, точность: 100.0%
count_children: ошибок: 0, точность: 100.0%
start_date: ошибок: 0, точность: 100.0%
nights: ошибок: 5, точность: 66.7%
price_per_day: ошибок: 1, точность: 93.3%
Средняя точность по всем показателям: 93.3%
